# 01｜DETR 概述与集合预测

前面已经学习了 CNN、Attention、Transformer Encoder、ViT 和 Swin Transformer。现在开始学习 DETR，把这些知识用于一个新任务：**目标检测**。

这一课只建立整体认识，不写模型代码，也不提前进入训练。重点是理解 DETR 想解决什么问题，以及它为什么把目标检测改写成集合预测。

## 1. 为什么现在学习 DETR

已经学过的内容正好组成 DETR 的前置基础：

- CNN：从图片中提取二维特征图。
- Attention：根据内容动态计算信息之间的关系。
- Transformer Encoder：让所有位置交换信息并获得全局上下文。
- ViT：把图像空间位置整理成 token 序列。
- Swin Transformer：理解多尺度视觉特征与高分辨率图像问题。

DETR 并不是把这些模块简单拼在一起。它更重要的贡献是：**用 Transformer 和集合预测重新组织目标检测流程。**

## 2. 先分清图像分类与目标检测

### 图像分类

图像分类回答：**这张图片主要是什么？**

例如输入一张图片，模型输出：猫。通常整张图片只需要一个类别结果。

### 目标检测

目标检测同时回答两个问题：

1. 图片里有哪些物体？
2. 每个物体在哪里？

因此，每个检测结果至少包含：

- 类别，例如人、狗、自行车。
- 边界框，例如中心位置、宽度和高度。

更关键的是，一张图片中的物体数量不固定。图片 A 可能有 2 个物体，图片 B 可能有 7 个物体，所以检测模型面对的是一个**变长、多目标输出问题**。

## 3. 传统目标检测流程为什么比较复杂

在 DETR 之前，许多检测器会使用下面这些专门设计：

- Anchor：预先在不同位置摆放多种大小和比例的参考框。
- 候选框：先产生大量可能包含物体的区域。
- 人工匹配规则：规定真实框应该由哪些 Anchor 负责。
- NMS：多个预测框重复检测同一个物体时，删除高度重叠的重复框。

可以把传统流程粗略理解成：

```text
图片
-> 生成大量候选预测
-> 判断类别并调整边界框
-> 按置信度排序
-> 使用 NMS 删除重复框
-> 得到最终检测结果
```

这些方法很有效，但整个系统包含较多任务专用组件和超参数。DETR 想问：**能不能直接让模型输出最终那一组互不重复的物体？**

## 4. DETR 是什么

DETR 的全称是 **DEtection TRansformer**。它把目标检测看成一个直接的集合预测问题。

原始 DETR 的主要结构是：

```text
输入图片
-> CNN Backbone 提取特征图
-> 1 x 1 卷积统一通道维度
-> 展平成空间 token 序列，并加入二维位置编码
-> Transformer Encoder 编码全局图像信息
-> Transformer Decoder 接收一组 Object Queries
-> 每个 Query 输出一个类别和一个边界框
-> 去掉预测为 no object 的槽位
-> 得到最终检测集合
```

这里仍然有 CNN 和 Transformer，但不再依赖 Anchor 和 NMS 组织最终输出。

## 5. DETR 的完整形状主线

设：

- batch 大小为 $B$。
- Backbone 输出通道数为 $C$，空间尺寸为 $H'\times W'$。
- Transformer 隐藏维度为 $D$。
- 空间 token 数为 $L=H'W'$。
- Object Query 数量为 $N$。
- 真实物体类别数为 $K$。

主要形状如下：

| 位置 | 形状 | 含义 |
|---|---|---|
| 输入图片 | $B\times3\times H\times W$ | 一批 RGB 图片 |
| CNN 特征图 | $B\times C\times H'\times W'$ | 带空间布局的视觉特征 |
| 通道投影后 | $B\times D\times H'\times W'$ | 对齐 Transformer 维度 |
| 展平后 | $B\times L\times D$ | 每个空间位置成为一个 token |
| Encoder 输出 | $B\times L\times D$ | 融合全局上下文的图像 memory |
| Decoder 输出 | $B\times N\times D$ | 每个 Query 得到一个预测表示 |
| 类别输出 | $B\times N\times(K+1)$ | $K$ 个物体类加 no object |
| 边界框输出 | $B\times N\times4$ | 每个 Query 的归一化边界框 |

边界框常写成 $(c_x,c_y,w,h)$，表示中心点横坐标、中心点纵坐标、宽度和高度，通常归一化到 0 至 1。

## 6. Encoder 和 Decoder 分别在做什么

### Encoder：理解整张图

Backbone 的每个空间位置先表示图片的一块区域。Encoder 使用 Self-Attention 让这些位置交换信息。

例如，一个只看到局部纹理的位置，可以结合远处的轮廓、上下文和其他物体，逐步形成更完整的场景表示。Encoder 输出通常称为 **memory**。

### Decoder：从图中取出一组物体

Decoder 接收一组 Object Queries。每个 Query 会：

1. 通过 Query 之间的 Self-Attention 交换信息。
2. 通过 Cross-Attention 从 Encoder memory 中寻找与自己有关的图像区域。
3. 经过多层更新后，形成一个用于预测类别和边界框的向量。

所以可以先把两者记成：

- Encoder：把整张图片编码成带全局上下文的视觉记忆。
- Decoder：用多个查询槽位，从视觉记忆中并行取出多个物体。

## 7. Object Query 到底是什么

Object Query 是 DETR 中最容易被误解的概念。

它不是：

- 某个固定类别，例如第 1 个 Query 永远负责猫。
- 某个固定物体，例如第 7 个 Query 永远负责图片中的第二个人。
- 一个预先放在确定位置和大小上的 Anchor。

更合适的理解是：**Object Query 是一个可学习的预测槽位。**

换成更直观的话说，100 个 Queries 就像 100 个固定答题框。每个答题框最多交出一个检测结果；图片里只有 3 个物体时，只需要 3 个答题框真正写答案，其余答题框写 `no object`。

从模型输出角度看，假设模型使用 100 个 Queries，就相当于准备了 100 个候选输出位置。每个槽位都尝试从图像 memory 中提取一个物体：

- 图片只有 3 个物体时，最终有少数槽位负责这 3 个物体。
- 其余槽位应该预测为 `no object`。

这些 Query 向量通过训练学会用不同方式询问图像，但单个 Query 的具体职责可以随图片变化。

## 8. 为什么叫集合预测

一张图片的真实标注可以写成：

$$
Y=\{(c_1,b_1),(c_2,b_2),\ldots,(c_M,b_M)\}
$$

其中 $c_i$ 是类别，$b_i$ 是边界框，$M$ 是这张图片中的真实物体数量。

这是一组物体，而不是一个有固定顺序的序列。把“人、狗、自行车”换成“狗、自行车、人”，标注含义没有变化。

模型却会输出固定数量 $N$ 个预测：

$$
\hat{Y}=\{(\hat{p}_1,\hat{b}_1),\ldots,(\hat{p}_N,\hat{b}_N)\}
$$

其中多余位置用 `no object` 表示。因此训练时的核心问题是：**哪一个预测应该和哪一个真实物体比较？**

## 9. 为什么需要匈牙利匹配

假设图片中有一只猫，模型的 100 个 Queries 中有 5 个都预测了猫。如果只是分别计算损失，不规定谁负责哪个目标，就可能出现多个槽位争抢同一个物体。

DETR 在计算最终损失前，先做一次**二分图一对一匹配**，通常使用匈牙利算法寻找总代价较小的配对。

匹配代价会综合考虑：

- 类别是否正确。
- 预测框与真实框的坐标距离。
- 两个框的重叠质量。

例如有 3 个真实物体时，只会选出 3 个不同的 Queries 分别与它们匹配；其他 Queries 统一学习预测 `no object`。

一对一匹配带来两个关键结果：

1. 一个真实物体只分配给一个预测槽位。
2. 模型直接学习产生互不重复的最终集合，因此推理时不再依赖 NMS 删除重复框。

## 10. 匹配完成后怎样计算损失

匹配决定“谁和谁比较”，损失函数再衡量预测有多大误差。主要包括：

### 分类损失

判断匹配到的 Query 是否给出了正确类别；没有匹配到真实物体的 Query 应该预测 `no object`。

### L1 边界框损失

比较预测框与真实框的 $(c_x,c_y,w,h)$ 坐标差异。

### GIoU 损失

从框的重叠和空间关系衡量定位质量，弥补只比较坐标或只使用普通 IoU 时的不足。

可以先记成：

$$
\mathcal{L}=\mathcal{L}_{cls}+\lambda_1\mathcal{L}_{L1}+\lambda_2\mathcal{L}_{GIoU}
$$

这里的系数用于平衡不同损失。具体匹配公式和 GIoU 计算留到后面的专门课程。

## 11. DETR 为什么可以称为端到端

这里的“端到端”主要表示：模型从图片直接学习输出最终物体集合，训练目标与最终预测形式保持一致。

它简化了传统检测器中的许多手工设计：

| 对比角度 | 传统检测器中的常见做法 | 原始 DETR |
|---|---|---|
| 初始候选 | 大量 Anchors 或候选区域 | 固定数量 Object Queries |
| 重复预测处理 | NMS 后处理 | 一对一集合预测 |
| 目标分配 | 多种人工匹配规则 | 全局二分图匹配 |
| 全局关系 | 依赖卷积层逐步扩大感受野 | Encoder/Decoder Attention |

但端到端不等于“什么组件都没有”。DETR 仍然需要 Backbone、位置编码、Transformer、预测头、匹配算法和损失函数。

## 12. DETR 与已经学过的知识怎样连接

### 与 CNN 的连接

原始 DETR 使用 CNN Backbone 提取特征图。CNN 负责把像素变成更有语义的视觉特征。

### 与 Attention 的连接

Encoder 的 Self-Attention 建立不同图像位置之间的关系；Decoder 的 Cross-Attention 让 Queries 从图像 memory 中寻找物体信息。

### 与 ViT 的连接

两者都会把二维特征整理成 token 序列并加入位置编码。不同的是，ViT 分类通常汇总成一个类别，而 DETR Decoder 需要输出多个物体槽位。

### 与 Swin Transformer 的连接

目标检测尤其需要空间细节和多尺度特征。学习 Swin 时建立的层级视觉特征观念，有助于以后理解为什么原始 DETR 对小目标不够理想，以及后续检测模型怎样改进多尺度处理。

## 13. 原始 DETR 的优点与局限

### 优点

- 把目标检测统一成集合预测问题。
- 不依赖 Anchor 和 NMS。
- 结构清晰，CNN 与 Transformer 职责明确。
- Attention 能显式利用全局图像关系。

### 局限

- 原始 DETR 训练收敛较慢。
- 对小目标的检测效果相对不足。
- Encoder 对展平后的高分辨率特征做全局 Attention，计算代价较高。

后续出现了 Deformable DETR 等改进方法，但现在不要急着跳过去。先把原始 DETR 的主线真正学懂。

## 14. 本节小结

这一课需要真正记住六个结论：

1. 目标检测要同时预测不定数量物体的类别和位置。
2. DETR 使用 CNN 提取特征，再用 Transformer 编码全局图像信息。
3. Object Query 是可学习的预测槽位，不是固定类别、固定物体或 Anchor。
4. 每个 Query 输出一个类别和一个边界框，多余槽位输出 `no object`。
5. 匈牙利匹配在预测集合和真实集合之间建立一对一对应。
6. 一对一集合预测让 DETR 不再需要 NMS 删除重复框。

最重要的数据流是：

```text
图片
-> CNN 特征图
-> 图像 tokens
-> Encoder memory
-> N 个 Object Queries
-> N 组类别与边界框
-> 去掉 no object
-> 最终目标集合
```

下一课再单独学习 DETR 输入如何从 CNN 特征图变成 Transformer tokens，以及二维位置编码为什么不可缺少。

## 15. 自测问题

1. 图像分类与目标检测的输出有什么不同？
2. 为什么目标检测是一个变长输出问题？
3. Anchor 和 NMS 在传统检测流程中分别做什么？
4. DETR 的 Backbone、Encoder、Decoder 分别负责什么？
5. DETR 为什么仍然需要位置编码？
6. Object Query 为什么不能简单理解成一个固定物体？
7. 使用 100 个 Queries 是否表示图片里必须有 100 个物体？
8. `no object` 类别解决了什么问题？
9. 为什么真实检测结果应该被看成无序集合？
10. 匈牙利匹配发生在计算损失之前还是之后？
11. 一对一匹配为什么能减少重复预测？
12. 类别输出为什么有 $K+1$ 个类别？
13. 边界框输出最后一维的 4 个值通常是什么？
14. 原始 DETR 为什么不需要 NMS？
15. 原始 DETR 的两个主要局限是什么？

### 自测参考答案

1. 分类通常给整张图一个类别；检测要为多个物体分别给出类别和边界框。
2. 不同图片包含的真实物体数量不同。
3. Anchor 提供预设参考框；NMS 删除多个高度重叠的重复预测框。
4. Backbone 提取视觉特征，Encoder 融合全局图像信息，Decoder 用 Queries 从 memory 中提取多个物体表示。
5. 展平后的 Attention 本身不知道 token 原来位于二维图像的哪里。
6. Query 是可学习的预测槽位，像一个固定答题框；它不是固定物体，具体负责对象可以随输入图片变化。
7. 不是。100 表示最多准备 100 个输出槽位，也就是 100 个候选答题框；多余槽位预测为 `no object`。
8. 它让固定数量的预测槽位能够表示图片中不固定数量的真实物体。
9. 物体排列顺序改变不会改变图片中的目标含义。
10. 先匹配，再根据匹配结果计算损失。
11. 每个真实目标只与一个不同的预测槽位匹配。
12. 除了 $K$ 个真实物体类别，还要增加 `no object` 类别。
13. 通常是归一化后的 $(c_x,c_y,w,h)$。
14. 一对一集合预测直接训练模型产生互不重复的最终结果。
15. 收敛较慢，并且对小目标的表现相对不足。